# Stage 5 — Temporal anomaly model (train on normal-only footage)

A small **GRU autoencoder** learns to reconstruct 16-frame windows of the
per-frame feature vectors produced by stage 4 (`data/features/*.npy`). It is
trained **only on normal launch footage**, so it learns the structure of a
typical ascent; at scoring time (stage 6) a high reconstruction error = anomaly.

**Critical check this notebook exists for.** Every video right now is tagged
`normal` — there is no anomaly footage to validate detection against yet. The
real test here is whether the 3 held-out normal videos score **similarly** to
the training videos. If the held-out videos score noticeably higher, the model
is overfitting to specific training videos rather than learning general
"normal ascent" structure, and that needs fixing *before* adding anomaly
footage is even worth doing.

**Held-out validation videos** (3 Starlink missions spread across lighting,
`SEED = 42`, recomputed deterministically in Cell 1 and asserted to match):

| Lighting | Held-out video |
|---|---|
| dusk | `spacex_starlink-20230303` |
| night | `spacex_starlink-20230822` |
| night | `spacex_starlink-20230901` |

**Split rationale.** `spacex_sda-tranche-0-2` is the only non-Starlink mission
in the dataset (its own camera/mission type, lighting=day). Holding it out
entirely means training never sees that camera type — a distribution shift in
the split, not a fair test of generalization. So it is **forced into
training**, and the 3 held-out videos are all Starlink missions, allocated
across the dusk (4 videos) and night (8 videos) clusters proportionally to
cluster size (1 dusk + 2 night).


In [ ]:
# --- Configuration (edit this cell for your environment) ---
import os, sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Drive folder that holds this repo's data/ + models/ dirs.
    DRIVE_BASE = '/content/drive/MyDrive/rocket-launch-anomaly-detector'
    LOCAL = '/content/rocket-launch-anomaly-detector'   # Colab local disk
else:
    # Local run: both point at the repo root (where this notebook lives).
    DRIVE_BASE = LOCAL = os.path.abspath('.')

FEATURES = Path(f'{LOCAL}/data/features')
MODELS = Path(f'{LOCAL}/models')
DRIVE_MODELS = Path(f'{DRIVE_BASE}/models')

SEED = 42
WINDOW = 16
STRIDE = 4
N_HOLD = 3

print('IN_COLAB =', IN_COLAB)
print('FEATURES =', FEATURES)
print('MODELS   =', MODELS)


### 0. Environment setup (Colab)

Clone the repo (or upload `scripts/` + `requirements.txt` into `LOCAL`
manually), install deps, copy the stage-4 feature files from Drive onto the
Colab local disk, and define the artifact push-back. Feature vectors are
KB-scale, so this whole stage runs comfortably on CPU.


In [ ]:
# --- Cell: install deps + make scripts/ importable (run once per session) ---
import shutil
import time

if not os.path.exists(f'{LOCAL}/scripts/features.py'):
    # Repo is missing or stale. Either set GIT_URL to your repo, or upload the
    # project (scripts/ + requirements.txt) into LOCAL manually and re-run.
    GIT_URL = 'https://github.com/YOUR_USERNAME/rocket-launch-anomaly-detector.git'
    if os.path.isdir(LOCAL) and os.listdir(LOCAL):
        stale = f'{LOCAL}_stale_{int(time.time())}'
        shutil.move(LOCAL, stale)
        print(f'moved existing {LOCAL} -> {stale}')
    elif os.path.isdir(LOCAL):
        os.rmdir(LOCAL)   # empty dir left by a previously failed clone
    print('cloning repo ...')
    !git clone --depth 1 {GIT_URL} "{LOCAL}"
    if not os.path.exists(f'{LOCAL}/scripts/features.py'):
        raise RuntimeError('clone produced no scripts/ — set GIT_URL to your repo '
                           'or upload the project into LOCAL')

!pip install -q -r "{LOCAL}/requirements.txt"

sys.path.insert(0, f'{LOCAL}/scripts')
import features as F     # only used for FEATURE_NAMES / sanity checks

import numpy as np
print('imports ok')


In [ ]:
# --- Cell: mount Drive + copy data/features locally + artifact push-back ---
if IN_COLAB:
    drive.mount('/content/drive')

import shutil

if not FEATURES.is_dir() or not any(FEATURES.glob('*.npy')):
    print('copying data/features from Drive to local disk ...')
    !cp -r "{DRIVE_BASE}/data/features" "{LOCAL}/data/"
else:
    print('features already on local disk:', FEATURES)

MODELS.mkdir(parents=True, exist_ok=True)
n_feats = len(list(FEATURES.glob('*.npy')))
print(f'{n_feats} feature files available')
assert n_feats > 0, 'no data/features/*.npy found — run notebook 04 first'


def push_artifacts():
    """Copy the trained scaler + model back to Drive (no-op in local mode)."""
    if DRIVE_MODELS == MODELS:
        return
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    for f in ('scaler.pkl', 'temporal_autoencoder.pt'):
        src = MODELS / f
        if src.exists():
            shutil.copy2(src, DRIVE_MODELS / f)
            print('pushed', f)


### 1. Data loading & standardization

Load all 13 feature sequences. Hold out 3 Starlink videos spread across
lighting clusters; the unique non-Starlink video (`spacex_sda-tranche-0-2`,
day) is **forced into training** so the model learns that camera/mission type.
**Standardize using only training-video statistics** — the scaler is fit on
training frames, applied to both train and validation, and saved to
`models/scaler.pkl`. Never fit scaler on validation data.


In [ ]:
# --- Cell 1: load sequences, hold out 3 validation videos, fit scaler on train ---
import json
import pickle
import numpy as np


def load_video(name):
    """Return (feature matrix [L, n_feat], video metadata dict)."""
    X = np.load(FEATURES / f'{name}.npy').astype(np.float32)
    idx = json.loads((FEATURES / f'{name}.frame_index.json').read_text())
    assert X.shape[1] == idx['n_features'] == F.N_FEATURES
    return X, idx['video']


metadata = {}
for npy in sorted(FEATURES.glob('*.npy')):
    name = npy.stem
    X, meta = load_video(name)
    metadata[name] = {'lighting': meta.get('lighting', 'unknown'),
                      'mission': meta.get('mission', name),
                      'frames': len(X)}

# --- hold out 3 Starlink videos, spread across lighting (fixed seed) ---
# The only non-Starlink video (spacex_sda-tranche-0-2, lighting=day) is a
# unique camera/mission type: forcing it into TRAINING so the model sees it.
# Held-out candidates are the Starlink videos; N_HOLD are allocated across
# lighting clusters proportional to cluster size (dusk 4 + night 8 -> 1 + 2).
starlink = {n: m for n, m in metadata.items()
            if m.get('mission', n).startswith('starlink')}
forced_train = sorted(set(metadata) - set(starlink))
assert len(forced_train) == 1, f'expected exactly 1 non-Starlink video: {forced_train}'

rng = np.random.default_rng(SEED)
clusters = {}
for n, m in starlink.items():
    clusters.setdefault(m['lighting'], []).append(n)

quota = {k: N_HOLD * len(v) / len(starlink) for k, v in clusters.items()}
alloc = {k: int(q) for k, q in quota.items()}
rem = N_HOLD - sum(alloc.values())
for k in sorted(quota, key=lambda k: quota[k] - alloc[k], reverse=True):
    if rem <= 0:
        break
    alloc[k] += 1
    rem -= 1

held_out = {}
for light in sorted(alloc):
    pool = sorted(clusters[light])
    held_out[light] = sorted(
        pool[i] for i in rng.choice(len(pool), size=alloc[light], replace=False))
held_out_names = sorted(n for v in held_out.values() for n in v)
train_names = sorted(n for n in metadata if n not in held_out_names)

EXPECTED_HOLD_OUT = ['spacex_starlink-20230303', 'spacex_starlink-20230822',
                     'spacex_starlink-20230901']
assert held_out_names == EXPECTED_HOLD_OUT, f'unexpected hold-out: {held_out_names}'
assert set(metadata) == set(train_names) | set(held_out_names)
assert forced_train[0] in train_names

print(f'Held-out validation ({len(held_out_names)}):')
for light in sorted(held_out):
    print(f'  {light:6s} -> {held_out[light]}')
print(f'Training ({len(train_names)}):')
for n in train_names:
    print('  ', n)

# --- standardize with statistics from TRAINING videos only ---
all_train = np.concatenate([load_video(n)[0] for n in train_names], axis=0)
scaler_mean = all_train.mean(axis=0)
scaler_std = all_train.std(axis=0)
scaler_std = np.where(scaler_std < 1e-6, 1.0, scaler_std)

with open(MODELS / 'scaler.pkl', 'wb') as f:
    pickle.dump({'mean': scaler_mean, 'std': scaler_std,
                 'feature_names': F.FEATURE_NAMES}, f)


def apply_scaler(X):
    return (X - scaler_mean) / scaler_std


print()
print(f'scaler fit on {all_train.shape[0]} TRAINING frames -> models/scaler.pkl')
print(f'per-feature mean range: {scaler_mean.min():.3f}..{scaler_mean.max():.3f}, '
      f'std range: {scaler_std.min():.3f}..{scaler_std.max():.3f}')


### 2. Windowing

Split every scaled sequence into overlapping windows of `WINDOW = 16` frames
with `STRIDE = 4`. Each video of length L yields about `(L - 16) / 4` windows;
windows are grouped per video so validation videos never leak into training.


In [ ]:
# --- Cell 2: window each scaled sequence into (WINDOW, n_features) windows ---
def make_windows(X):
    starts = range(0, len(X) - WINDOW + 1, STRIDE)
    return np.stack([X[s:s + WINDOW] for s in starts])


train_windows = {n: make_windows(apply_scaler(load_video(n)[0])) for n in train_names}
val_windows = {n: make_windows(apply_scaler(load_video(n)[0])) for n in held_out_names}

print('train windows:')
for n in train_names:
    print(f'  {n:<38} {train_windows[n].shape}')
print('held-out windows:')
for n in held_out_names:
    print(f'  {n:<38} {val_windows[n].shape}')

n_tr = sum(w.shape[0] for w in train_windows.values())
n_va = sum(w.shape[0] for w in val_windows.values())
print(f'\ntotal windows: {n_tr} train, {n_va} held-out')


### 3. Model definition

Small GRU autoencoder: GRU encoder → latent vector → GRU decoder (teacher
forced) → linear head. Reconstructs the input window; training loss is MSE
reconstruction error. Deliberately tiny — a small model on a small dataset.


In [ ]:
# --- Cell 3: define the GRU autoencoder (small, no over-engineering) ---
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(SEED)


class WindowDataset(Dataset):
    """Concatenated window tensors from a dict of video -> windows."""

    def __init__(self, windows):
        self.X = torch.from_numpy(np.concatenate(list(windows.values())))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i]


class GRUAutoencoder(nn.Module):
    """Encode a window into a latent vector, then reconstruct it.

    Encoder: GRU over the window -> last hidden state -> latent z.
    Decoder: GRU seeded with z and teacher-forced on the (shifted) input,
    with a linear output head back to feature space.
    """

    def __init__(self, input_dim=33, hidden=64, latent=32):
        super().__init__()
        self.encoder = nn.GRU(input_dim, hidden, batch_first=True)
        self.fc_enc = nn.Linear(hidden, latent)
        self.fc_dec = nn.Linear(latent, hidden)
        self.decoder = nn.GRU(input_dim, hidden, batch_first=True)
        self.fc_out = nn.Linear(hidden, input_dim)

    def forward(self, x):
        _, h = self.encoder(x)                       # [1, B, hidden]
        z = torch.tanh(self.fc_enc(h[-1]))           # [B, latent]
        h0 = torch.tanh(self.fc_dec(z)).unsqueeze(0)  # [1, B, hidden]
        src = torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], dim=1)
        out, _ = self.decoder(src, h0)               # [B, T, hidden]
        return self.fc_out(out)                      # [B, T, input_dim]


model = GRUAutoencoder(input_dim=F.N_FEATURES)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\ntrainable parameters: {n_params:,}')


### 4. Training

Plain PyTorch training loop with Adam + MSE. Early stopping on **held-out
validation** reconstruction loss (patience 8, max 60 epochs), keeping the best
state. Saves `models/scaler.pkl` (already done in Cell 1) and
`models/temporal_autoencoder.pt`, then pushes both to Drive if in Colab.


In [ ]:
# --- Cell 4: training loop with early stopping on held-out reconstruction loss ---
import copy
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_dl = DataLoader(WindowDataset(train_windows), batch_size=256, shuffle=True)
val_dl = DataLoader(WindowDataset(val_windows), batch_size=256, shuffle=False)

MAX_EPOCHS, PATIENCE = 60, 8
best_val, best_state, patience = float('inf'), None, 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for xb in train_dl:
        xb = xb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), xb)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * len(xb)
    tr_loss /= len(train_dl.dataset)

    model.eval()
    with torch.no_grad():
        val_loss = sum(criterion(model(xb.to(device)), xb.to(device)).item() * len(xb)
                       for xb in val_dl) / len(val_dl.dataset)

    history.append((tr_loss, val_loss))
    if val_loss < best_val:
        best_val, best_state, patience = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f'early stop at epoch {epoch} (best held-out loss {best_val:.5f})')
            break
    print(f'epoch {epoch:3d}  train {tr_loss:.5f}  held-out {val_loss:.5f}')

model.load_state_dict(best_state)
torch.save({'state_dict': model.state_dict(),
            'input_dim': F.N_FEATURES, 'hidden': 64, 'latent': 32,
            'window': WINDOW, 'stride': STRIDE},
           MODELS / 'temporal_autoencoder.pt')
print(f'\nsaved models/temporal_autoencoder.pt (best held-out loss {best_val:.5f})')
push_artifacts()


### 5. Evaluation

For every video (train and held-out), compute a **per-frame reconstruction
error**: each frame's error is the mean MSE over all windows that cover it
(window 16 / stride 4 ⇒ each frame is covered by ~4 windows). Then compare the
held-out videos' error distribution against the training videos' distribution
side by side — the overfitting check.


In [ ]:
# --- Cell 5a: per-frame reconstruction error for every video (train + held-out) ---
device_for_eval = device


def per_frame_error(name):
    """Mean reconstruction MSE per frame, averaged over all windows covering it."""
    X = apply_scaler(load_video(name)[0])
    W = make_windows(X)
    model.eval()
    with torch.no_grad():
        recon = model(torch.from_numpy(W).to(device)).cpu().numpy()
    window_err = ((W - recon) ** 2).mean(axis=(1, 2))   # [n_windows]
    per_frame = np.zeros(len(X), np.float32)
    cover = np.zeros(len(X), np.int32)
    for i, s in enumerate(range(0, len(X) - WINDOW + 1, STRIDE)):
        per_frame[s:s + WINDOW] += window_err[i]
        cover[s:s + WINDOW] += 1
    return per_frame / np.maximum(1, cover)


errors = {}
for name in sorted(metadata):
    errors[name] = per_frame_error(name)

print('per-video mean reconstruction error (lower = more "normal"):')
print(f'  {"video":<38}{"mean":>10}{"p95":>10}{"max":>10}')
for name in sorted(metadata):
    e = errors[name]
    tag = 'HELD-OUT' if name in held_out_names else ''
    print(f'  {name:<38}{e.mean():>10.5f}{np.percentile(e, 95):>10.5f}'
          f'{e.max():>10.5f}  {tag}')


In [ ]:
# --- Cell 5b: reconstruction error over time for each video ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(metadata), 1, figsize=(13, 1.15 * len(metadata)),
                         sharex=True)
for ax, name in zip(axes, sorted(metadata)):
    e = errors[name]
    color = 'tab:red' if name in held_out_names else 'tab:blue'
    ax.plot(e, lw=0.6, color=color)
    ax.set_ylabel(name.replace('spacex_', ''), fontsize=8)
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)
    if name in held_out_names:
        ax.set_title(f'{name}  (HELD-OUT)', fontsize=8, color='tab:red',
                     loc='left', pad=1)
axes[-1].set_xlabel('frame')
fig.suptitle('Per-frame reconstruction error — blue: train, red: held-out',
             fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 5c: held-out vs training error distribution (the overfitting check) ---
train_err = np.concatenate([errors[n] for n in train_names])
val_err = np.concatenate([errors[n] for n in held_out_names])

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(train_err, bins=60, density=True, alpha=0.6, color='tab:blue',
           label='train (10 videos)')
ax[0].hist(val_err, bins=60, density=True, alpha=0.6, color='tab:red',
           label='held-out (3 videos)')
ax[0].set_xlabel('per-frame reconstruction error')
ax[0].set_ylabel('density')
ax[0].legend()
ax[0].set_title('Error distribution')
ax[1].boxplot([train_err, val_err], tick_labels=['train', 'held-out'])
ax[1].set_ylabel('per-frame reconstruction error')
ax[1].set_title('Side by side')
plt.tight_layout()
plt.show()

print(f'{"":>12}{"median":>10}{"mean":>10}{"p95":>10}{"max":>10}')
print(f'{"train":>12}{np.median(train_err):>10.5f}{train_err.mean():>10.5f}'
      f'{np.percentile(train_err, 95):>10.5f}{train_err.max():>10.5f}')
print(f'{"held-out":>12}{np.median(val_err):>10.5f}{val_err.mean():>10.5f}'
      f'{np.percentile(val_err, 95):>10.5f}{val_err.max():>10.5f}')

ratio = np.median(val_err) / max(1e-9, np.median(train_err))
print(f'\nheld-out/train median error ratio: {ratio:.2f}')
if ratio > 2.0:
    print('WARNING: held-out normals score much higher than training videos — '
          'the model is overfitting to specific training videos. Fix before '
          'adding anomaly footage (more data, stronger bottleneck, more '
          'regularization, or a wider validation split).')
else:
    print('OK: held-out normals score like training normals — the model is '
          'learning general ascent structure, not memorizing training videos.')
